# Chapter 10
## The Slow-Fast Phase Plane
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter10.ipynb)

## About this chapter

When one state changes much faster than another, trajectories move quickly
between branches of a slow manifold and linger near slow branches.
Nullclines locate the directions with zero velocity, while a closed orbit
describes repeated firing. A reduction with instantaneous sodium activation
and $h+n=0.83$ makes this geometry visible in a two-dimensional HH system.

The FitzHugh-Nagumo example uses

$$
\dot v=v-v^3/3-n+I,\qquad \dot n=(av-n)/\tau_n.
$$

Here $v$ is the fast voltage-like variable, $n$ is the slow recovery
variable, $I$ is applied current, $a$ sets the recovery nullcline, and
$\tau_n$ makes recovery slow. The reduced HH examples take $m=m_\infty(v)$
and $h=0.83-n$, leaving $(v,n)$ as the phase plane.

See [`README.md`](chapter10.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from mnd.core import draw_arrow

## FitzHugh-Nagumo Phase Plane

In [ ]:
def simulate_fn(a=1.25, tau_n=15.625, i_ext=-0.5, t_final=400.0, dt=0.01):
    def derivative(x0, t):
        v, n = x0
        dv = v - v ** 3 / 3 - n + i_ext
        dn = (a * v - n) / tau_n
        return [dv, dn]

    x0 = [-1.0, -2.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 1]


def plot_fn(t, v, n, a=1.25, i_ext=-0.5):
    v_line = np.arange(-100, 101) / 100.0 * 3
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

    ax[0].plot(v_line, a * v_line, color='k', linewidth=2)
    ax[0].plot(v_line, v_line - v_line ** 3 / 3 + i_ext, color='r', linewidth=2)
    ax[0].plot(v, n, color='b', linewidth=2)
    ax[0].set_xlim(-3, 3)
    ax[0].set_ylim(-3, 3)
    ax[0].set_xlabel('$v$')
    ax[0].set_ylabel('$n$')

    ax[1].plot(t, v, color='k', linewidth=2)
    ax[1].set_xlim(0, t[-1])
    ax[1].set_ylim(-3, 3)
    ax[1].set_xlabel('$t$')
    ax[1].set_ylabel('$v$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_fn(*simulate_fn())

In [ ]:
interact(lambda i_ext=-0.5: plot_fn(*simulate_fn(i_ext=i_ext), i_ext=i_ext),
         i_ext=(-1.5, 1.5, 0.05));

## Reduced HH v-Nullcline

Shared bisection solver for the reduced HH model's v-nullcline (the curve
in $(v,n)$ where the total membrane current is zero), used by both
`HH_CYCLE_SPEED` and `HH_NULLCLINES_PLUS_SOLUTION` below.

In [ ]:
def hh_v_nullcline(m_inf, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0, i_ext=10.0):
    v_vec = np.arange(-100, 51)
    n_vec = np.zeros_like(v_vec, dtype=float)
    for ij, v in enumerate(v_vec):
        n_l, n_r = 0.0, 1.0
        while n_r - n_l > 1e-10:
            n = (n_l + n_r) / 2
            i_ion = (g_na * m_inf(v) ** 3 * (0.83 - n) * (v_na - v)
                     + g_k * n ** 4 * (v_k - v) + g_l * (v_l - v) + i_ext)
            if i_ion < 0:
                n_r = n
            else:
                n_l = n
        n_vec[ij] = (n_l + n_r) / 2
        if n_vec[ij] > 1 - 1e-10:
            n_vec[ij] = 2
        if n_vec[ij] < 1e-10:
            n_vec[ij] = -1
    return v_vec, n_vec

## HH Limit-Cycle Speed

Trajectory speed in the $(v,n)$ reduced-HH phase plane, split into "fast"
and "slow" segments to show how the limit cycle spends most of its time
near the slow branches.

In [ ]:
def simulate_hh_cycle_speed(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                             v_k=-82.0, v_na=45.0, v_l=-59.0, i_ext=10.0):
    def alpha_m(v):
        if abs(v + 45) > 1e-8:
            return (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))
        return 1.0

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)

    def beta_m(v):
        return 4 * exp(-(v + 70) / 18)

    def beta_n(v):
        return 0.125 * exp(-(v + 70) / 80)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    v_vec, n_vec = hh_v_nullcline(m_inf, g_na, g_k, g_l, v_na, v_k, v_l, i_ext)

    t_final = 150.0
    dt = 0.001
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    speed = np.zeros(m_steps)
    v[0], n[0] = -70.0, 0.0

    for k in range(m_steps):
        m_k = m_inf(v[k])
        h_k = 0.83 - n[k]
        v_inc = (g_na * m_k ** 3 * h_k * (v_na - v[k]) + g_k * n[k] ** 4 * (v_k - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        v_tmp = v[k] + dt05 * v_inc
        n_tmp = n[k] + dt05 * n_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = 0.83 - n_tmp
        v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        v[k + 1] = v[k] + dt * v_inc
        n[k + 1] = n[k] + dt * n_inc
        speed[k] = np.sqrt((v_inc / 150) ** 2 + (n_inc / 0.35) ** 2)

    n_start = round(2 / 3 * m_steps)
    return v_vec, n_vec, v[n_start:m_steps], n[n_start:m_steps], speed[n_start:m_steps], n_inf


def plot_hh_cycle_speed(v_vec, n_vec, v, n, speed, n_inf):
    plt.figure(figsize=(7, 7))
    v_line = v_vec.astype(float) + 0.001
    plt.plot(v_line, n_inf(v_line), color='k', linewidth=3)
    plt.plot(v_vec, n_vec, color='r', linewidth=3)

    plt.plot(v, n, color='b', linewidth=2)
    fast = speed > speed.max() * 0.02
    plt.plot(v[fast], n[fast], '.b', markersize=10)
    plt.plot(v[~fast], n[~fast], '.g', markersize=10)

    plt.xlim(-100, 50)
    plt.ylim(0, 1)
    plt.xlabel('$v$ [mV]')
    plt.ylabel('$n$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_cycle_speed(*simulate_hh_cycle_speed())

## HH Nullclines Plus Solution

Same nullcline construction, overlaid with a single trajectory and
direction arrows.

In [ ]:
def simulate_hh_nullclines_plus_solution(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                                          v_k=-82.0, v_na=45.0, v_l=-59.0,
                                          i_ext=10.0, t_final=50.0, dt=0.01):
    def alpha_m(v):
        if abs(v + 45) > 1e-8:
            return (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))
        return 1.0

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)

    def beta_m(v):
        return 4 * exp(-(v + 70) / 18)

    def beta_n(v):
        return 0.125 * exp(-(v + 70) / 80)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative(x0, t):
        v, n = x0
        m = m_inf(v)
        h = 0.83 - n
        dv = (g_na * m ** 3 * h * (v_na - v) + g_k * n ** 4 * (v_k - v)
              + g_l * (v_l - v) + i_ext) / c
        dn = alpha_n(v) * (1 - n) - beta_n(v) * n
        return [dv, dn]

    v_vec, n_vec = hh_v_nullcline(m_inf, g_na, g_k, g_l, v_na, v_k, v_l, i_ext)

    x0 = [-70.0, 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return v_vec, n_vec, t, sol[:, 0], sol[:, 1], sol, n_inf


def plot_hh_nullclines_plus_solution(v_vec, n_vec, t, v, n, sol, n_inf, dt=0.01):
    fig, ax = plt.subplots(figsize=(7, 7))
    v_line = v_vec.astype(float) + 0.001
    ax.plot(v_line, n_inf(v_line), color='k', linewidth=3)
    ax.plot(v_vec, n_vec, color='r', linewidth=3)
    ax.plot(v, n, color='b', linewidth=2)

    for t_arrow in [0.57, 2.5, 6.5, 12.75, 13.2]:
        i = round(t_arrow / dt)
        vec = sol[i] - sol[i - 11]
        draw_arrow(ax, [-100, 50], [0, 1], v[i], n[i], vec, epsilon=0.05, color='b')

    ax.set_xlim(-100, 50)
    ax.set_ylim(0, 1)
    ax.set_xlabel('$v$ [mV]')
    ax.set_ylabel('$n$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_nullclines_plus_solution(*simulate_hh_nullclines_plus_solution())

## HH $h+n$ Approximation

Checks how closely the full HH model's $h+n$ stays near the reduced
model's constant $0.83$ assumption.

In [ ]:
def simulate_hh_h_plus_n(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                          v_k=-82.0, v_na=45.0, v_l=-59.0,
                          i_ext=10.0, t_final=50.0, dt=0.01):
    def alpha_h(v):
        return 0.07 * exp(-(v + 70) / 20)

    def alpha_m(v):
        return (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)

    def beta_h(v):
        return 1. / (exp(-(v + 40) / 10) + 1)

    def beta_m(v):
        return 4 * exp(-(v + 70) / 18)

    def beta_n(v):
        return 0.125 * exp(-(v + 70) / 80)

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative(x0, t):
        v, m, n, h = x0
        i_na = -g_na * h * m ** 3 * (v - v_na)
        i_k = -g_k * n ** 4 * (v - v_k)
        i_l = -g_l * (v - v_l)
        dv = (i_ext + i_na + i_k + i_l) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dm, dn, dh]

    v0 = -20.0
    x0 = [v0, m_inf(v0), n_inf(v0), h_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 2], sol[:, 3]


def plot_hh_h_plus_n(t, n, h):
    plt.figure(figsize=(7, 3))
    plt.axhline(y=0.83, ls="--", c="r", lw=2)
    plt.plot(t, h + n, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.ylim(0.7, 1)
    plt.xlabel("time [ms]", fontsize=14)
    plt.ylabel("h + n", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_h_plus_n(*simulate_hh_h_plus_n())

## Reduced HH Voltage Trace

The full HH model, reduced with $m=m_\infty(v)$ and $h=0.83-n$, leaving
$(v,n)$ as the state.

In [ ]:
def simulate_reduced_hh(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                         v_k=-82.0, v_na=45.0, v_l=-59.0,
                         i_ext=10.0, t_final=50.0, dt=0.01):
    def alpha_m(v):
        return (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)

    def beta_m(v):
        return 4 * exp(-(v + 70) / 18)

    def beta_n(v):
        return 0.125 * exp(-(v + 70) / 80)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def derivative(x0, t):
        v, n = x0
        m = m_inf(v)
        h = 0.83 - n
        i_na = -g_na * h * m ** 3 * (v - v_na)
        i_k = -g_k * n ** 4 * (v - v_k)
        i_l = -g_l * (v - v_l)
        dv = (i_ext + i_na + i_k + i_l) / c
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        return [dv, dn]

    v0 = -50.0
    x0 = [v0, 0.4]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 1], m_inf


def plot_reduced_hh(t, v, n, m_inf):
    fig, ax = plt.subplots(2, figsize=(7, 5), sharex=True)
    ax[0].plot(t, v, lw=2, c="k")
    ax[1].plot(t, n, lw=2, c="r", label="n")
    ax[1].plot(t, 0.83 - n, lw=2, c="g", label="h")
    ax[1].plot(t, m_inf(v), lw=2, c="b", label="m")

    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylim(-100, 50)
    ax[1].set_ylim(0, 1)
    ax[1].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("v [mV]", fontsize=14)
    ax[1].set_ylabel("m, h, n", fontsize=14)
    ax[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_reduced_hh(*simulate_reduced_hh())

In [ ]:
interact(lambda i_ext=10.0: plot_reduced_hh(*simulate_reduced_hh(i_ext=i_ext)),
         i_ext=(0.0, 20.0, 0.5));